# EDA — MVTec AD + HSS-IAD (Unified Dataset)

**Pré-requis :** exécuter `uv run python -m src.data.harmonize` pour générer les CSV.

**Sommaire :**
1. Chargement & vue d'ensemble
2. Distribution des images par catégorie et dataset
3. Ratio normal / anomal par catégorie
4. Distribution des résolutions d'images
5. Exemples visuels (normal vs anomal + masque)
6. Analyse des masques — ratio de pixels anomaux
7. Heatmap des types de défauts
8. Analyse de la diversité des canaux (RGB vs Grayscale)
9. Synthèse & recommandations

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Rendre la racine du projet importable quand le notebook est lancé
# depuis le dossier notebooks/
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA, EDA, PATHS

warnings.filterwarnings("ignore")
sns.set_theme(style=EDA.sns_style, palette=EDA.sns_palette, font_scale=EDA.sns_font_scale)
plt.rcParams["figure.dpi"] = EDA.figure_dpi

df = pd.read_csv(PATHS.unified_csv)
df_res = pd.read_csv(PATHS.resolutions_csv)

print(f"Dataset unifié : {len(df):,} images | {df['category'].nunique()} catégories | {df['dataset'].nunique()} sources")
df.head()

## 1. Vue d'ensemble

In [ ]:
# Résumé global par dataset
overview = df.groupby("dataset").agg(
    images=("image_path", "count"),
    categories=("category", "nunique"),
    anomalies=("is_anomaly", "sum"),
    masks=("has_mask", "sum"),
).assign(
    normal=lambda x: x["images"] - x["anomalies"],
    anomaly_pct=lambda x: (x["anomalies"] / x["images"] * 100).round(1),
    mask_coverage_pct=lambda x: (x["masks"] / x["anomalies"] * 100).round(1),
)
print(overview.to_string())

print("\n--- Détail par catégorie et split ---")
detail = df.groupby(["dataset", "category", "split"]).agg(
    total=("image_path", "count"),
    anomalies=("is_anomaly", "sum"),
    masks=("has_mask", "sum"),
).reset_index()
detail["normal"] = detail["total"] - detail["anomalies"]
detail

## 2. Distribution des images par catégorie et dataset

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

cat_counts = df.groupby(["category", "dataset"]).size().unstack(fill_value=0)
cat_counts = cat_counts.sort_values(by=cat_counts.columns.tolist(), ascending=False)

cat_counts.plot(kind="barh", stacked=True, ax=ax, color=[EDA.color_mvtec, EDA.color_hssiad])
ax.set_xlabel("Nombre d'images")
ax.set_title("Distribution des images par catégorie et dataset")
ax.legend(title="Dataset")
ax.invert_yaxis()

for container in ax.containers:
    ax.bar_label(container, label_type="center", fontsize=8, color="white", fmt="%d")

plt.tight_layout()
plt.show()

## 3. Ratio normal / anomal par catégorie

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

status_counts = df.groupby(["category", "is_anomaly"]).size().unstack(fill_value=0)
status_counts.columns = ["Normal", "Anomal"]

# Normalize to percentage
status_pct = status_counts.div(status_counts.sum(axis=1), axis=0) * 100
status_pct = status_pct.sort_values("Anomal", ascending=False)

status_pct.plot(kind="barh", stacked=True, ax=ax, color=[EDA.color_normal, EDA.color_anomal])
ax.set_xlabel("Pourcentage (%)")
ax.set_title("Ratio Normal / Anomal par catégorie")
ax.legend(title="Status")
ax.invert_yaxis()
ax.axvline(x=50, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Distribution des résolutions d'images

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Width vs Height scatter
for ds, color in [(DATA.mvtec_name, EDA.color_mvtec), (DATA.hssiad_name, EDA.color_hssiad)]:
    mask = df_res["dataset"] == ds
    axes[0].scatter(df_res.loc[mask, "width"], df_res.loc[mask, "height"],
                    alpha=0.5, label=ds, color=color, s=20)
axes[0].set_xlabel("Largeur (px)")
axes[0].set_ylabel("Hauteur (px)")
axes[0].set_title("Résolutions : Largeur vs Hauteur")
axes[0].legend()

# Resolution bar chart
df_res["resolution"] = df_res["width"].astype(int).astype(str) + "x" + df_res["height"].astype(int).astype(str)
res_counts = df_res.groupby(["resolution", "dataset"]).size().unstack(fill_value=0)
res_counts = res_counts.loc[res_counts.sum(axis=1).nlargest(10).index]
res_counts.plot(kind="barh", stacked=True, ax=axes[1], color=[EDA.color_mvtec, EDA.color_hssiad])
axes[1].set_xlabel("Nombre d'images (échantillon)")
axes[1].set_title("Top 10 résolutions")
axes[1].legend(title="Dataset")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\nRésolutions uniques par dataset :")
for ds in df_res["dataset"].unique():
    resolutions = df_res[df_res["dataset"] == ds]["resolution"].unique()
    print(f"  {ds}: {sorted(resolutions)}")

## 5. Exemples visuels — Normal vs Anomal + Masque

Pour chaque dataset, on affiche quelques catégories avec : image normale, image anomale, masque associé.

In [ ]:
import re

# Pour HSS-IAD Casting, chaque pièce est photographiée sous 3 angles
# encodés en suffixe (_1_2, _1_3, _2_3). On apparie Normal et Anomal
# sur le même suffixe pour que les exemples soient comparables.
_VIEW_RE = re.compile(r"_(\d+_\d+)$")


def _view_key(stem: str) -> str | None:
    m = _VIEW_RE.search(stem)
    return m.group(1) if m else None


def _overlay_mask(img: Image.Image, mask: Image.Image, color=(255, 0, 0), alpha=0.45) -> Image.Image:
    """Superpose un masque coloré semi-transparent sur une image RGB."""
    base = img.convert("RGBA")
    mask_arr = np.array(mask.convert("L"))
    overlay = np.zeros((*mask_arr.shape, 4), dtype=np.uint8)
    overlay[mask_arr > 0] = [*color, int(255 * alpha)]
    overlay_img = Image.fromarray(overlay, mode="RGBA")
    return Image.alpha_composite(base, overlay_img).convert("RGB")


def show_samples(df, dataset_name, n_categories=EDA.n_example_categories, seed=EDA.example_seed):
    """Display normal / anomal / mask / overlay quadruplets for selected categories.

    Quand les noms de fichiers contiennent un suffixe de vue (ex. `_1_2`),
    on s'assure que l'image normale et l'image anomale partagent la même
    vue — indispensable pour HSS-IAD Casting_class{1,2,3}.
    Toutes les images sont converties en RGB pour uniformiser l'affichage.
    """
    rng = np.random.RandomState(seed)
    sub = df[df["dataset"] == dataset_name]
    categories = rng.choice(sub["category"].unique(),
                            size=min(n_categories, sub["category"].nunique()),
                            replace=False)

    fig, axes = plt.subplots(len(categories), 4, figsize=(16, 4 * len(categories)))
    if len(categories) == 1:
        axes = axes[np.newaxis, :]

    for i, cat in enumerate(categories):
        cat_df = sub[sub["category"] == cat]

        # 1) On tire d'abord l'image anomale (avec masque)
        anomal = cat_df[(cat_df["is_anomaly"] == True) & (cat_df["has_mask"] == True)]
        anom_row = None
        view = None
        if len(anomal) > 0:
            anom_row = anomal.iloc[rng.randint(len(anomal))]
            view = _view_key(Path(anom_row["image_path"]).stem)

        # 2) On choisit un normal avec la MÊME vue si possible
        normal = cat_df[(cat_df["is_anomaly"] == False) & (cat_df["split"] == "test")]
        if len(normal) == 0:
            normal = cat_df[cat_df["is_anomaly"] == False]
        if view is not None:
            same_view = normal[normal["image_path"].apply(
                lambda p: _view_key(Path(p).stem) == view
            )]
            if len(same_view) > 0:
                normal = same_view

        # --- Affichage Normal ---
        if len(normal) > 0:
            img_path = PATHS.root / normal.iloc[rng.randint(len(normal))]["image_path"]
            axes[i, 0].imshow(Image.open(img_path).convert("RGB"))
        title_view = f" (vue {view})" if view else ""
        axes[i, 0].set_title(f"{cat} — Normal{title_view}")
        axes[i, 0].axis("off")

        # --- Affichage Anomal + masque + overlay ---
        if anom_row is not None:
            anom_img = Image.open(PATHS.root / anom_row["image_path"]).convert("RGB")
            mask_img = Image.open(PATHS.root / anom_row["mask_path"])

            axes[i, 1].imshow(anom_img)
            axes[i, 1].set_title(f"{cat} — {anom_row['label']}{title_view}")

            axes[i, 2].imshow(mask_img, cmap="Reds")
            axes[i, 2].set_title(f"{cat} — Masque")

            axes[i, 3].imshow(_overlay_mask(anom_img, mask_img))
            axes[i, 3].set_title(f"{cat} — Défaut + masque")
        else:
            axes[i, 1].set_title(f"{cat} — (pas d'anomalie)")
            axes[i, 2].set_title(f"{cat} — (pas de masque)")
            axes[i, 3].set_title(f"{cat} — (pas d'overlay)")

        axes[i, 1].axis("off")
        axes[i, 2].axis("off")
        axes[i, 3].axis("off")

    fig.suptitle(f"Exemples — {dataset_name}", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()


show_samples(df, DATA.mvtec_name)
show_samples(df, DATA.hssiad_name)

## 6. Analyse des masques — Ratio de pixels anomaux

On mesure, pour chaque image anomale avec masque, le pourcentage de pixels défectueux. Cela donne une idée de la **taille des défauts** par catégorie.

In [ ]:
# Compute anomaly pixel ratio on masks (sample for speed)
masks_df = df[df["has_mask"] == True].copy()
sample_masks = masks_df.sample(n=min(800, len(masks_df)), random_state=DATA.random_seed)

anomaly_ratios = []
for _, row in sample_masks.iterrows():
    try:
        mask = np.array(Image.open(PATHS.root / row["mask_path"]).convert("L"))
        ratio = (mask > 0).sum() / mask.size * 100
        anomaly_ratios.append(ratio)
    except Exception:
        anomaly_ratios.append(None)

sample_masks = sample_masks.copy()
sample_masks["anomaly_pixel_pct"] = anomaly_ratios

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot by dataset
sample_masks.boxplot(column="anomaly_pixel_pct", by="dataset", ax=axes[0])
axes[0].set_title("Ratio pixels anomaux par dataset")
axes[0].set_ylabel("% pixels anomaux")
axes[0].set_xlabel("")
plt.sca(axes[0])
plt.title("Ratio pixels anomaux par dataset")

# Boxplot by category (top 10 most représentées)
top_cats = sample_masks["category"].value_counts().head(10).index
sub = sample_masks[sample_masks["category"].isin(top_cats)]
sub.boxplot(column="anomaly_pixel_pct", by="category", ax=axes[1], rot=45)
axes[1].set_title("Ratio pixels anomaux (top 10 catégories)")
axes[1].set_ylabel("% pixels anomaux")
axes[1].set_xlabel("")

fig.suptitle("")
plt.tight_layout()
plt.show()

print(f"Ratio moyen global : {sample_masks['anomaly_pixel_pct'].mean():.2f}%")
print(f"Ratio médian global : {sample_masks['anomaly_pixel_pct'].median():.2f}%")
print(f"\nPar dataset :")
print(sample_masks.groupby("dataset")["anomaly_pixel_pct"].describe().round(2).to_string())

## 7. Heatmap des types de défauts par catégorie

In [ ]:
# Defect type heatmap
anomalies = df[df["is_anomaly"] == True]
defect_matrix = anomalies.groupby(["category", "label"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(defect_matrix, annot=True, fmt="d", cmap="YlOrRd", ax=ax, linewidths=0.5)
ax.set_title("Nombre d'images par type de défaut et catégorie")
ax.set_xlabel("Type de défaut")
ax.set_ylabel("Catégorie")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"Types de défauts uniques : {anomalies['label'].nunique()}")
print(f"Catégories avec anomalies : {anomalies['category'].nunique()}")

## 8. Analyse des canaux (RGB vs Grayscale)

In [ ]:
df_res["color_mode"] = df_res["channels"].map({1: "Grayscale", 3: "RGB", 4: "RGBA"}).fillna("Autre")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Par dataset
df_res.groupby(["dataset", "color_mode"]).size().unstack(fill_value=0).plot(
    kind="bar", ax=axes[0], color=["#7B7B7B", EDA.color_mvtec, EDA.color_hssiad]
)
axes[0].set_title("Canaux par dataset")
axes[0].set_ylabel("Nombre d'images (échantillon)")
axes[0].tick_params(axis='x', rotation=0)

# Par catégorie
channel_by_cat = df_res.groupby(["category", "color_mode"]).size().unstack(fill_value=0)
channel_by_cat.plot(kind="barh", stacked=True, ax=axes[1], color=["#7B7B7B", EDA.color_mvtec, EDA.color_hssiad])
axes[1].set_title("Canaux par catégorie")
axes[1].set_xlabel("Nombre d'images (échantillon)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("Catégories en Grayscale :")
gray_cats = df_res[df_res["color_mode"] == "Grayscale"]["category"].unique()
print(f"  {sorted(gray_cats) if len(gray_cats) > 0 else '(aucune)'}")

## 9. Synthèse & Recommandations

In [ ]:
# Automated summary
print("=" * 60)
print("SYNTHÈSE")
print("=" * 60)

print(f"\n📊 Volume total : {len(df):,} images")
for ds in df["dataset"].unique():
    sub = df[df["dataset"] == ds]
    n_anom = sub["is_anomaly"].sum()
    n_mask = sub["has_mask"].sum()
    print(f"  {ds}: {len(sub):,} images | {sub['category'].nunique()} catégories | "
          f"{n_anom:,} anomalies | {n_mask:,} masques")

print(f"\n🔍 Couverture masques :")
total_anom = df["is_anomaly"].sum()
total_mask = df["has_mask"].sum()
missing = total_anom - total_mask
print(f"  {total_mask:,}/{total_anom:,} anomalies ont un masque ({total_mask/total_anom*100:.1f}%)")
if missing > 0:
    missing_cats = df[(df["is_anomaly"]) & (~df["has_mask"])].groupby(["dataset", "category"]).size()
    print(f"  ⚠ {missing:,} masques manquants :")
    print(missing_cats.to_string())

print(f"\n📐 Résolutions :")
for ds in df_res["dataset"].unique():
    resolutions = df_res[df_res["dataset"] == ds]["resolution"].unique()
    print(f"  {ds}: {len(resolutions)} résolution(s) uniques — {sorted(resolutions)[:5]}")

print("\n" + "=" * 60)
print("RECOMMANDATIONS POUR LE PIPELINE")
print("=" * 60)
print("""
1. RÉSOLUTION : Standardiser toutes les images à une taille commune 
   (ex: 256x256 ou 224x224) via resize + center crop.

2. CANAUX : Convertir toutes les images en RGB (les grayscale seront 
   dupliquées sur 3 canaux).

3. MASQUES MANQUANTS : Vérifier les catégories avec masques manquants. 
   Si trop de masques manquent, ces catégories ne pourront servir que 
   pour l'évaluation image-level (pas pixel-level).

4. DÉSÉQUILIBRE : Le ratio normal/anomal varie fortement entre catégories.
   Pour l'entraînement (non-supervisé), seules les images "good" du split 
   train sont utilisées — pas de problème. Pour l'évaluation, le 
   déséquilibre est normal en anomaly detection.

5. SPLIT : Garder les splits train/test originaux. Ne pas mélanger 
   pour préserver la validité des benchmarks.
""")